In [2]:
import polars as pl
import torch 
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
import numpy as np
import random
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report
import seaborn as sns
from tqdm import tqdm
import warnings
warnings.filterwarnings('ignore')

print("Device:", torch.device("cuda" if torch.cuda.is_available() else "cpu"))
torch.manual_seed(42)
np.random.seed(42)
random.seed(42)

Device: cuda


## READ AND PREPROCESS

In [3]:
df = pl.read_csv("../data/train.csv")

In [4]:
# Train/test split (80/20) respetando series temporales
# Dividir por sequence_id para mantener la proporción 80/20

random.seed(42)

# Obtener los sequence_ids únicos
sequence_ids = df.select("sequence_id").unique().to_series().to_list()
random.shuffle(sequence_ids)

# Dividir sequence_ids 80/20
split_idx = int(len(sequence_ids) * 0.8)
train_sequence_ids = sequence_ids[:split_idx]
test_sequence_ids = sequence_ids[split_idx:]

# Filtrar el dataframe según los sequence_ids
train_df = df.filter(df["sequence_id"].is_in(train_sequence_ids))
test_df = df.filter(df["sequence_id"].is_in(test_sequence_ids))

print(f"Train sequence_ids: {len(train_sequence_ids)}, rows: {train_df.shape[0]}")
print(f"Test sequence_ids: {len(test_sequence_ids)}, rows: {test_df.shape[0]}")
print(f"Total sequence_ids: {len(sequence_ids)}, rows: {df.shape[0]}")
print(f"Train ratio: {train_df.shape[0] / df.shape[0] * 100:.1f}%")


Train sequence_ids: 6520, rows: 458984
Test sequence_ids: 1631, rows: 115961
Total sequence_ids: 8151, rows: 574945
Train ratio: 79.8%


In [5]:
# Revisar NaNs en train_df y test_df
print("=== Train DataFrame ===")
print(f"Total rows: {train_df.shape[0]}")
print(f"NaNs por columna:")
print(train_df.null_count())

print("\n=== Test DataFrame ===")
print(f"Total rows: {test_df.shape[0]}")
print(f"NaNs por columna:")
print(test_df.null_count())

# Verificar si hay filas con al menos un NaN
train_with_nulls = train_df.filter(pl.any_horizontal(pl.col("*").is_null()))
test_with_nulls = test_df.filter(pl.any_horizontal(pl.col("*").is_null()))

print(f"\n=== Filas con NaN ===")
print(f"Train: {train_with_nulls.shape[0]} filas con al menos un NaN")
print(f"Test: {test_with_nulls.shape[0]} filas con al menos un NaN")

=== Train DataFrame ===
Total rows: 458984
NaNs por columna:
shape: (1, 341)
┌────────┬────────────┬────────────┬───────────┬───┬───────────┬───────────┬───────────┬───────────┐
│ row_id ┆ sequence_t ┆ sequence_i ┆ sequence_ ┆ … ┆ tof_5_v60 ┆ tof_5_v61 ┆ tof_5_v62 ┆ tof_5_v63 │
│ ---    ┆ ype        ┆ d          ┆ counter   ┆   ┆ ---       ┆ ---       ┆ ---       ┆ ---       │
│ u32    ┆ ---        ┆ ---        ┆ ---       ┆   ┆ u32       ┆ u32       ┆ u32       ┆ u32       │
│        ┆ u32        ┆ u32        ┆ u32       ┆   ┆           ┆           ┆           ┆           │
╞════════╪════════════╪════════════╪═══════════╪═══╪═══════════╪═══════════╪═══════════╪═══════════╡
│ 0      ┆ 0          ┆ 0          ┆ 0         ┆ … ┆ 25470     ┆ 25470     ┆ 25470     ┆ 25470     │
└────────┴────────────┴────────────┴───────────┴───┴───────────┴───────────┴───────────┴───────────┘

=== Test DataFrame ===
Total rows: 115961
NaNs por columna:
shape: (1, 341)
┌────────┬────────────┬────────────┬──

In [6]:
# === FASE 1: CORRECCIÓN DE CARGA Y LIMPIEZA ===

# 1. Agrupar sensores ToF en arrays (preserva estructura espacial 8x8)
def group_tof_sensors(lf: pl.LazyFrame) -> pl.LazyFrame:
    """Agrupa los 64 píxeles de cada sensor ToF en un array,
    transformando 5x64 columnas escalares en 5 columnas de arrays."""
    for i in range(1, 6):
        tof_cols = [f"tof_{i}_v{j}" for j in range(64)]
        lf = lf.with_columns(pl.concat_arr(tof_cols).alias(f"tof_{i}"))
    return lf.drop(pl.selectors.matches(r"^tof_\d_v\d+$"))

# 2. Función mejorada de limpieza que maneja TODAS las columnas float
def clean_sequences_v2(df):
    """
    Limpia NaNs en cada sequence_id usando estrategia robusta:
    - Agrupa sensores ToF en arrays (preserva estructura)
    - Interpola + forward/backward fill en cada secuencia
    - Rellena valores restantes con media global por columna
    - Filtra secuencias que pierdan >50% de datos
    - Preserva metadatos
    """
    # Convertir a LazyFrame, agrupar ToF, y volver a DataFrame
    df = group_tof_sensors(df.lazy()).collect()
    
    # Identificar columnas float escalares (excluye arrays)
    float_cols = [name for name, dtype in df.schema.items() 
                  if dtype in [pl.Float32, pl.Float64]]
    
    # Calcular medias globales para rellenar valores residuales
    global_means = df.select([pl.col(c).mean().alias(f"{c}_mean") for c in float_cols])
    means_dict = {col: global_means[f"{col}_mean"][0] for col in float_cols}
    
    cleaned_dfs = []
    valid_seq_ids = []
    
    for seq_id in df.select("sequence_id").unique().to_series():
        seq_df = df.filter(df["sequence_id"] == seq_id).sort("sequence_counter")
        original_len = len(seq_df)
        
        # Estrategia 1: Interpolar + forward fill + backward fill
        seq_df = seq_df.with_columns([
            pl.col(c).interpolate()
                    .forward_fill()
                    .backward_fill()
            for c in float_cols
        ])
        
        # Estrategia 2: Rellenar NaNs restantes con media global
        seq_df = seq_df.with_columns([
            pl.col(c).fill_null(means_dict[c])
            for c in float_cols
        ])
        
        # Estrategia 3: Filtrar secuencias que pierdan >50% de filas válidas
        remaining_len = len(seq_df.filter(pl.any_horizontal([pl.col(c).is_not_null() for c in float_cols])))
        if remaining_len >= original_len * 0.5:  # Al menos 50% válido
            cleaned_dfs.append(seq_df)
            valid_seq_ids.append(seq_id)
    
    result = pl.concat(cleaned_dfs) if cleaned_dfs else df.slice(0, 0)
    return result

# Aplicar limpieza
print("Limpiando train_df...")
train_df_cleaned = clean_sequences_v2(train_df)
print("Limpiando test_df...")
test_df_cleaned = clean_sequences_v2(test_df)

print(f"\n=== LIMPIEZA COMPLETADA ===")
print(f"Train shape: {train_df_cleaned.shape}")
print(f"Test shape: {test_df_cleaned.shape}")
print(f"Train NaNs totales: {int(train_df_cleaned.null_count().select(pl.col('*')).sum_horizontal()[0])}")
print(f"Test NaNs totales: {int(test_df_cleaned.null_count().select(pl.col('*')).sum_horizontal()[0])}")
print(f"Columnas después de agrupar ToF: {train_df_cleaned.columns}")
print(f"\n✓ Datos limpios y listos para procesamiento")

Limpiando train_df...
Limpiando test_df...

=== LIMPIEZA COMPLETADA ===
Train shape: (458984, 26)
Test shape: (115961, 26)
Train NaNs totales: 0
Test NaNs totales: 0
Columnas después de agrupar ToF: ['row_id', 'sequence_type', 'sequence_id', 'sequence_counter', 'subject', 'orientation', 'behavior', 'phase', 'gesture', 'acc_x', 'acc_y', 'acc_z', 'rot_w', 'rot_x', 'rot_y', 'rot_z', 'thm_1', 'thm_2', 'thm_3', 'thm_4', 'thm_5', 'tof_1', 'tof_2', 'tof_3', 'tof_4', 'tof_5']

✓ Datos limpios y listos para procesamiento


## FASE 2: PREPARACIÓN PARA MomentFM

In [7]:
# Análisis exploratorio
print("=== ANÁLISIS EXPLORATORIO ===")
print(f"\nGestures únicos: {train_df_cleaned.select('gesture').unique().shape[0]}")
print(f"Distribución de gestures en train:")
print(train_df_cleaned.group_by("gesture").agg(pl.len().alias("count")).sort("count", descending=True))

print(f"\nSujetos únicos: {train_df_cleaned.select('subject').unique().shape[0]}")
print(f"Orientaciones únicas: {train_df_cleaned.select('orientation').unique().shape[0]}")
print(f"Fases únicas: {train_df_cleaned.select('phase').unique().shape[0]}")

print(f"\nSecuencias por longitud:")
seq_lengths = train_df_cleaned.group_by("sequence_id").agg(pl.len().alias("length"))
print(seq_lengths.describe())

=== ANÁLISIS EXPLORATORIO ===

Gestures únicos: 18
Distribución de gestures en train:
shape: (18, 2)
┌───────────────────────┬───────┐
│ gesture               ┆ count │
│ ---                   ┆ ---   │
│ str                   ┆ u32   │
╞═══════════════════════╪═══════╡
│ Text on phone         ┆ 46843 │
│ Neck - scratch        ┆ 44536 │
│ Eyebrow - pull hair   ┆ 35997 │
│ Forehead - scratch    ┆ 33143 │
│ Cheek - pinch skin    ┆ 32841 │
│ …                     ┆ …     │
│ Drink from bottle/cup ┆ 10734 │
│ Scratch knee/leg skin ┆ 10675 │
│ Glasses on/off        ┆ 10484 │
│ Write name on leg     ┆ 8387  │
│ Pinch knee/leg skin   ┆ 7929  │
└───────────────────────┴───────┘

Sujetos únicos: 81
Orientaciones únicas: 4
Fases únicas: 2

Secuencias por longitud:
shape: (9, 3)
┌────────────┬─────────────┬───────────┐
│ statistic  ┆ sequence_id ┆ length    │
│ ---        ┆ ---         ┆ ---       │
│ str        ┆ str         ┆ f64       │
╞════════════╪═════════════╪═══════════╡
│ count      ┆ 6

In [8]:
# Codificar target (gesture -> índice numérico)
gesture_encoder = LabelEncoder()

# Entrenar encoder en todos los valores únicos de train_df_cleaned
unique_gestures = train_df_cleaned.select("gesture").unique().to_series().to_list()
gesture_encoder.fit(unique_gestures)

# Aplicar en train
train_df_cleaned = train_df_cleaned.with_columns(
    pl.col("gesture").map_elements(
        lambda x: int(gesture_encoder.transform([x])[0]),
        return_dtype=pl.Int32
    ).alias("gesture_encoded")
)

# Aplicar en test
test_df_cleaned = test_df_cleaned.with_columns(
    pl.col("gesture").map_elements(
        lambda x: int(gesture_encoder.transform([x])[0]),
        return_dtype=pl.Int32
    ).alias("gesture_encoded")
)

# Mapping legible
gesture_mapping = {idx: name for idx, name in enumerate(gesture_encoder.classes_)}
print(f"\n=== MAPEO DE GESTURES ===")
for idx, name in sorted(gesture_mapping.items()):
    print(f"{idx}: {name}")

num_classes = len(gesture_encoder.classes_)
print(f"\nTotal de clases: {num_classes}")


=== MAPEO DE GESTURES ===
0: Above ear - pull hair
1: Cheek - pinch skin
2: Drink from bottle/cup
3: Eyebrow - pull hair
4: Eyelash - pull hair
5: Feel around in tray and pull out an object
6: Forehead - pull hairline
7: Forehead - scratch
8: Glasses on/off
9: Neck - pinch skin
10: Neck - scratch
11: Pinch knee/leg skin
12: Pull air toward your face
13: Scratch knee/leg skin
14: Text on phone
15: Wave hello
16: Write name in air
17: Write name on leg

Total de clases: 18


In [ ]:
# Restructurar datos a formato (num_sequences, timesteps, features)
def restructure_to_sequences(df_cleaned, use_tof_arrays=True):
    """
    Convierte DataFrame (sequence_id, sequence_counter, features) a
    formato (num_sequences, timesteps_variable, num_features)
    
    Features: 7 IMU + 5 Termopila + 320 ToF (5 arrays de 64 píxeles) = 332 features
    """
    imu_cols = ["acc_x", "acc_y", "acc_z", "rot_w", "rot_x", "rot_y", "rot_z"]
    thm_cols = ["thm_1", "thm_2", "thm_3", "thm_4", "thm_5"]
    tof_cols = ["tof_1", "tof_2", "tof_3", "tof_4", "tof_5"] if use_tof_arrays else []
    
    all_feature_cols = imu_cols + thm_cols + tof_cols
    
    sequences_data = []
    sequence_ids = []
    labels = []
    
    for seq_id in df_cleaned.select("sequence_id").unique().to_series():
        seq_df = df_cleaned.filter(df_cleaned["sequence_id"] == seq_id).sort("sequence_counter")
        
        # Extraer features
        if use_tof_arrays:
            # Para arrays ToF, convertir cada array a lista de floats
            seq_features = []
            for row in seq_df.select(all_feature_cols).iter_rows(named=True):
                row_features = []
                try:
                    # Procesar IMU + Termopila
                    for col in imu_cols + thm_cols:
                        val = row[col]
                        # Manejar valores None/NaN
                        if val is None or (isinstance(val, float) and np.isnan(val)):
                            row_features.append(0.0)
                        else:
                            row_features.append(float(val))
                    
                    # Procesar ToF arrays
                    for col in tof_cols:
                        tof_array = row[col]
                        if tof_array is not None:
                            for x in tof_array:
                                if x == -1 or x is None:
                                    row_features.append(0.0)
                                else:
                                    row_features.append(float(x))
                    
                    seq_features.append(row_features)
                except Exception as e:
                    print(f"Error procesando fila de secuencia {seq_id}: {e}")
                    continue
        else:
            seq_features = seq_df.select(all_feature_cols).to_numpy(allow_copy=True)
        
        if len(seq_features) > 0:
            sequences_data.append(np.array(seq_features, dtype=np.float32))
            sequence_ids.append(seq_id)
            # Obtener el label (todos los rows de una secuencia tienen el mismo gesture_encoded)
            label = seq_df.select("gesture_encoded").to_series()[0]
            labels.append(int(label))
    
    return sequences_data, np.array(labels, dtype=np.int64), sequence_ids

print("\nRestructurando datos...")
train_sequences, train_labels, train_seq_ids = restructure_to_sequences(train_df_cleaned, use_tof_arrays=True)
test_sequences, test_labels, test_seq_ids = restructure_to_sequences(test_df_cleaned, use_tof_arrays=True)

print(f"Train: {len(train_sequences)} secuencias")
print(f"  Shape primera secuencia: {train_sequences[0].shape} (timesteps, features)")
print(f"Test: {len(test_sequences)} secuencias")
print(f"  Shape primera secuencia: {test_sequences[0].shape} (timesteps, features)")
print(f"\nTrain labels shape: {train_labels.shape}, dtype: {train_labels.dtype}")
print(f"Test labels shape: {test_labels.shape}, dtype: {test_labels.dtype}")

# Verificar número de features
num_features = train_sequences[0].shape[1]
print(f"\n✓ Número total de features: {num_features}")
print(f"  Composición: 7 IMU + 5 Termopila + 320 ToF (5×64) = {7+5+320}")
print(f"  Match: {num_features == 332}")


Restructurando datos...


In [ ]:
# Normalizar features
print("\nNormalizando features...")

# Concatenar todas las secuencias para calcular estadísticas
all_train_data = np.vstack(train_sequences)

# Inicializar scaler
scaler = StandardScaler()
scaler.fit(all_train_data)

# Aplicar normalización a todas las secuencias
train_sequences_normalized = []
for seq in train_sequences:
    normalized = scaler.transform(seq)
    train_sequences_normalized.append(normalized)

test_sequences_normalized = []
for seq in test_sequences:
    normalized = scaler.transform(seq)
    test_sequences_normalized.append(normalized)

print(f"Train normalized: {len(train_sequences_normalized)} secuencias")
print(f"Test normalized: {len(test_sequences_normalized)} secuencias")
print(f"Mean de train[0] (debe ser ~0): {train_sequences_normalized[0].mean():.6f}")
print(f"Std de train[0] (debe ser ~1): {train_sequences_normalized[0].std():.6f}")

In [ ]:
# Crear DataLoaders PyTorch con padding
print("\nCreando DataLoaders...")

def pad_sequences(sequences, max_length=None):
    """Rellena secuencias de longitud variable con ceros al final."""
    if max_length is None:
        max_length = max(len(seq) for seq in sequences)
    
    padded = []
    masks = []
    for seq in sequences:
        if len(seq) < max_length:
            # Padding con ceros
            padded_seq = np.pad(seq, ((0, max_length - len(seq)), (0, 0)), mode='constant')
            mask = np.concatenate([np.ones(len(seq)), np.zeros(max_length - len(seq))])
        else:
            padded_seq = seq[:max_length]
            mask = np.ones(max_length)
        padded.append(padded_seq)
        masks.append(mask)
    
    return np.array(padded), np.array(masks)

# Encontrar longitud máxima
max_len_train = max(len(seq) for seq in train_sequences_normalized)
max_len_test = max(len(seq) for seq in test_sequences_normalized)
max_length = max(max_len_train, max_len_test)
print(f"Longitud máxima de secuencias: {max_length}")

# Rellenar secuencias
train_padded, train_masks = pad_sequences(train_sequences_normalized, max_length)
test_padded, test_masks = pad_sequences(test_sequences_normalized, max_length)

print(f"Train padded shape: {train_padded.shape} (secuencias, timesteps, features)")
print(f"Train masks shape: {train_masks.shape}")

# Convertir a tensores PyTorch
train_X = torch.from_numpy(train_padded).float()
train_y = torch.from_numpy(train_labels).long()
train_masks_t = torch.from_numpy(train_masks).float()

test_X = torch.from_numpy(test_padded).float()
test_y = torch.from_numpy(test_labels).long()
test_masks_t = torch.from_numpy(test_masks).float()

# Crear DataLoaders
BATCH_SIZE = 16
train_loader = DataLoader(
    TensorDataset(train_X, train_y, train_masks_t),
    batch_size=BATCH_SIZE,
    shuffle=True
)

test_loader = DataLoader(
    TensorDataset(test_X, test_y, test_masks_t),
    batch_size=BATCH_SIZE,
    shuffle=False
)

print(f"\nTrain loader: {len(train_loader)} batches de {BATCH_SIZE}")
print(f"Test loader: {len(test_loader)} batches de {BATCH_SIZE}")

# Verificar un batch
X_batch, y_batch, mask_batch = next(iter(train_loader))
print(f"\nBatch X shape: {X_batch.shape}")
print(f"Batch y shape: {y_batch.shape}")
print(f"Batch mask shape: {mask_batch.shape}")

## FASE 3: MODELO Y FINE-TUNING

In [ ]:
# Intentar importar MomentFM o crear modelo alternativo
try:
    from momentfm.models import MomentModel
    print("✓ MomentFM disponible")
    USING_MOMENTFM = True
except ImportError:
    print("✗ MomentFM no disponible, usando modelo Transformer personalizado")
    USING_MOMENTFM = False

if not USING_MOMENTFM:
    # Crear modelo Transformer personalizado para series temporales
    class TransformerTimeSeriesClassifier(nn.Module):
        """Clasificador Transformer optimizado para series temporales (similar a MomentFM)"""
        def __init__(self, input_size, hidden_size=128, num_heads=4, num_layers=2, num_classes=10, dropout=0.1):
            super().__init__()
            self.input_size = input_size
            
            # Linear projection de features a hidden_size
            self.input_projection = nn.Linear(input_size, hidden_size)
            
            # Encoder Transformer
            encoder_layer = nn.TransformerEncoderLayer(
                d_model=hidden_size,
                nhead=num_heads,
                dim_feedforward=hidden_size * 4,
                dropout=dropout,
                batch_first=True,
                activation='relu'
            )
            self.transformer_encoder = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)
            
            # Cabeza de clasificación
            self.classifier = nn.Sequential(
                nn.Linear(hidden_size, hidden_size // 2),
                nn.ReLU(),
                nn.Dropout(dropout),
                nn.Linear(hidden_size // 2, num_classes)
            )
        
        def forward(self, x, mask=None):
            """
            x: (batch_size, seq_len, input_size)
            mask: (batch_size, seq_len) - 1 para timesteps válidos, 0 para padding
            """
            batch_size, seq_len, _ = x.shape
            
            # Proyectar features
            x = self.input_projection(x)  # (batch_size, seq_len, hidden_size)
            
            # Crear máscara de padding para Transformer
            if mask is not None:
                # Convertir máscara de 1s/0s a máscara de Transformer (True para ignorar)
                transformer_mask = (mask == 0).to(x.device)
            else:
                transformer_mask = None
            
            # Aplicar Transformer encoder
            x = self.transformer_encoder(x, src_key_padding_mask=transformer_mask)
            
            # Pooling: usar solo timesteps válidos
            if mask is not None:
                # Mean pooling sobre timesteps válidos
                mask_expanded = mask.unsqueeze(2).expand(x.size())  # (batch_size, seq_len, hidden_size)
                x_masked = x * mask_expanded
                x_pooled = x_masked.sum(dim=1) / mask_expanded.sum(dim=1).clamp(min=1)
            else:
                x_pooled = x.mean(dim=1)
            
            # Clasificación
            logits = self.classifier(x_pooled)
            return logits

# Configuración del modelo
model_config = {
    'input_size': num_features,
    'hidden_size': 128,
    'num_heads': 4,
    'num_layers': 2,
    'num_classes': num_classes,
    'dropout': 0.1
}

if USING_MOMENTFM:
    # TODO: Configurar MomentFM si está disponible
    print("Configurando MomentFM...")
else:
    model = TransformerTimeSeriesClassifier(**model_config)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)

print(f"\n=== MODELO INSTANCIADO ===")
print(f"Device: {device}")
print(f"Número de parámetros: {sum(p.numel() for p in model.parameters()):,}")
print(f"Configuración: {model_config}")

In [ ]:
# Fine-tuning del modelo
print("\n=== INICIANDO FINE-TUNING ===\n")

# Configuración de entrenamiento
NUM_EPOCHS = 20
LEARNING_RATE = 1e-4
WEIGHT_DECAY = 1e-5
PATIENCE = 3  # Early stopping

# Criterio de pérdida y optimizador
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode='max', factor=0.5, patience=2, verbose=True
)

# Historial de entrenamiento
history = {
    'train_loss': [],
    'train_acc': [],
    'test_acc': [],
    'test_f1': []
}

best_test_acc = 0
patience_counter = 0

# Training loop
for epoch in range(NUM_EPOCHS):
    # === TRAIN ===
    model.train()
    train_loss = 0
    train_preds = []
    train_targets = []
    
    for X_batch, y_batch, mask_batch in tqdm(train_loader, desc=f"Epoch {epoch+1}/{NUM_EPOCHS} [Train]"):
        X_batch = X_batch.to(device)
        y_batch = y_batch.to(device)
        mask_batch = mask_batch.to(device)
        
        # Forward pass
        logits = model(X_batch, mask_batch)
        loss = criterion(logits, y_batch)
        
        # Backward pass
        optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        
        # Tracking
        train_loss += loss.item()
        train_preds.extend(logits.argmax(dim=1).cpu().numpy())
        train_targets.extend(y_batch.cpu().numpy())
    
    train_loss /= len(train_loader)
    train_acc = accuracy_score(train_targets, train_preds)
    
    # === TEST ===
    model.eval()
    test_preds = []
    test_targets = []
    
    with torch.no_grad():
        for X_batch, y_batch, mask_batch in test_loader:
            X_batch = X_batch.to(device)
            y_batch = y_batch.to(device)
            mask_batch = mask_batch.to(device)
            
            logits = model(X_batch, mask_batch)
            test_preds.extend(logits.argmax(dim=1).cpu().numpy())
            test_targets.extend(y_batch.cpu().numpy())
    
    test_acc = accuracy_score(test_targets, test_preds)
    
    # Registrar historial
    history['train_loss'].append(train_loss)
    history['train_acc'].append(train_acc)
    history['test_acc'].append(test_acc)
    
    # Learning rate scheduler
    scheduler.step(test_acc)
    
    print(f"Epoch {epoch+1}/{NUM_EPOCHS} | Loss: {train_loss:.4f} | "
          f"Train Acc: {train_acc:.4f} | Test Acc: {test_acc:.4f}")
    
    # Early stopping
    if test_acc > best_test_acc:
        best_test_acc = test_acc
        patience_counter = 0
        best_model_state = model.state_dict().copy()
    else:
        patience_counter += 1
        if patience_counter >= PATIENCE:
            print(f"\nEarly stopping en epoch {epoch+1}")
            model.load_state_dict(best_model_state)
            break

print(f"\n✓ Entrenamiento completado")
print(f"Mejor Test Accuracy: {best_test_acc:.4f}")

## FASE 4: EVALUACIÓN Y RESULTADOS

In [ ]:
# Evaluación completa en test set
print("\n=== EVALUACIÓN FINAL ===\n")

model.eval()
test_preds_final = []
test_targets_final = []
test_probs_final = []

with torch.no_grad():
    for X_batch, y_batch, mask_batch in test_loader:
        X_batch = X_batch.to(device)
        y_batch = y_batch.to(device)
        mask_batch = mask_batch.to(device)
        
        logits = model(X_batch, mask_batch)
        probs = torch.softmax(logits, dim=1)
        
        test_preds_final.extend(logits.argmax(dim=1).cpu().numpy())
        test_targets_final.extend(y_batch.cpu().numpy())
        test_probs_final.extend(probs.cpu().numpy())

test_preds_final = np.array(test_preds_final)
test_targets_final = np.array(test_targets_final)
test_probs_final = np.array(test_probs_final)

# Calcular métricas
test_accuracy = accuracy_score(test_targets_final, test_preds_final)
cm = confusion_matrix(test_targets_final, test_preds_final)

print(f"Test Accuracy: {test_accuracy:.4f}")
print(f"\nClassification Report:")
print(classification_report(
    test_targets_final, 
    test_preds_final,
    target_names=[gesture_mapping[i] for i in range(num_classes)],
    zero_division=0
))

In [ ]:
# Visualizaciones
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# 1. Training curves
axes[0, 0].plot(history['train_loss'], label='Train Loss', marker='o')
axes[0, 0].set_xlabel('Epoch')
axes[0, 0].set_ylabel('Loss')
axes[0, 0].set_title('Training Loss')
axes[0, 0].legend()
axes[0, 0].grid()

# 2. Accuracy curves
axes[0, 1].plot(history['train_acc'], label='Train Acc', marker='o')
axes[0, 1].plot(history['test_acc'], label='Test Acc', marker='s')
axes[0, 1].set_xlabel('Epoch')
axes[0, 1].set_ylabel('Accuracy')
axes[0, 1].set_title('Training vs Test Accuracy')
axes[0, 1].legend()
axes[0, 1].grid()

# 3. Confusion matrix
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=axes[1, 0], 
            xticklabels=[gesture_mapping[i][:20] for i in range(num_classes)],
            yticklabels=[gesture_mapping[i][:20] for i in range(num_classes)])
axes[1, 0].set_title('Confusion Matrix')
axes[1, 0].set_ylabel('True Label')
axes[1, 0].set_xlabel('Predicted Label')

# 4. Accuracy por clase
class_accuracies = cm.diagonal() / cm.sum(axis=1)
sorted_idx = np.argsort(class_accuracies)
sorted_gestures = [gesture_mapping[i][:30] for i in sorted_idx]
sorted_accs = class_accuracies[sorted_idx]

axes[1, 1].barh(sorted_gestures, sorted_accs, color='steelblue')
axes[1, 1].set_xlabel('Accuracy')
axes[1, 1].set_title('Accuracy por Gesture')
axes[1, 1].set_xlim([0, 1])

plt.tight_layout()
plt.savefig('model_evaluation.png', dpi=150, bbox_inches='tight')
plt.show()

print("✓ Gráficos guardados en 'model_evaluation.png'")

In [ ]:
# Análisis de errores
print("\n=== ANÁLISIS DE ERRORES ===\n")

# Encontrar predicciones incorrectas
incorrect_mask = test_preds_final != test_targets_final
incorrect_count = incorrect_mask.sum()
print(f"Total de predicciones incorrectas: {incorrect_count}/{len(test_targets_final)}")

if incorrect_count > 0:
    print("\nTop 5 confusiones:")
    incorrect_indices = np.where(incorrect_mask)[0]
    
    # Analizar confusiones
    confusion_pairs = {}
    for idx in incorrect_indices:
        true_label = test_targets_final[idx]
        pred_label = test_preds_final[idx]
        pair = (gesture_mapping[true_label], gesture_mapping[pred_label])
        confusion_pairs[pair] = confusion_pairs.get(pair, 0) + 1
    
    # Top 5
    for (true, pred), count in sorted(confusion_pairs.items(), key=lambda x: x[1], reverse=True)[:5]:
        print(f"  {true} → {pred}: {count} veces")

# Accuracy por difficulty
print("\n=== ESTADÍSTICAS POR GESTURE ===")
for i in range(num_classes):
    mask = test_targets_final == i
    if mask.sum() > 0:
        acc = (test_preds_final[mask] == i).mean()
        count = mask.sum()
        print(f"{gesture_mapping[i][:40]:40s} - Acc: {acc:.3f} ({count} muestras)")

In [ ]:
# Resumen ejecutivo
print("\n" + "="*60)
print("RESUMEN EJECUTIVO")
print("="*60)

print(f"""
📊 DATASET:
  • Train sequences: {len(train_sequences_normalized)}
  • Test sequences: {len(test_sequences_normalized)}
  • Total features: {num_features} (7 IMU + 5 Termopila + 320 ToF)
  • Número de gestures: {num_classes}
  • Longitud max: {max_length} timesteps
  • Batch size: {BATCH_SIZE}

🧠 MODELO:
  • Arquitectura: Transformer Encoder (similar a MomentFM)
  • Parámetros: {sum(p.numel() for p in model.parameters()):,}
  • Hidden size: {model_config['hidden_size']}
  • Num heads: {model_config['num_heads']}
  • Num layers: {model_config['num_layers']}

📈 ENTRENAMIENTO:
  • Epochs completados: {len(history['train_loss'])}
  • Learning rate: {LEARNING_RATE}
  • Mejor train loss: {min(history['train_loss']):.6f}
  • Mejor train accuracy: {max(history['train_acc']):.4f}

✅ RESULTADOS FINALES:
  • Test Accuracy: {test_accuracy:.4f}
  • Baseline aleatorio: {1/num_classes:.4f}
  • Mejora vs baseline: {(test_accuracy - 1/num_classes)*100:.1f}%
  • Predicciones correctas: {(test_preds_final == test_targets_final).sum()}/{len(test_targets_final)}
  • Predicciones incorrectas: {incorrect_count}/{len(test_targets_final)}

🎯 TOP 3 GESTURES CON MEJOR ACCURACY:
""")

class_accuracies_sorted = sorted(
    [(gesture_mapping[i], cm[i,i] / max(cm[i].sum(), 1)) for i in range(num_classes)],
    key=lambda x: x[1],
    reverse=True
)

for gesture, acc in class_accuracies_sorted[:3]:
    print(f"  1. {gesture[:50]:50s} - {acc:.3f}")

print(f"\n{'='*60}")
print(f"Estado: ✓ Modelo entrenado y evaluado exitosamente")
print(f"{'='*60}")